In [184]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from difflib import get_close_matches
import pickle


In [185]:
CHUNK_SIZE = 100_000

def load_large_csv(path):
    chunks = []
    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE):
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)


In [186]:
books   = load_large_csv("/content/Books.csv")
ratings = load_large_csv("/content/Ratings.csv")
users   = load_large_csv("/content/Users.csv")


In [187]:
books.drop_duplicates(subset="ISBN", inplace=True)
ratings = ratings[ratings["Book-Rating"] > 0]

df = ratings.merge(books, on="ISBN")
df = df[df["User-ID"].isin(users["User-ID"])]


In [188]:
active_users = df["User-ID"].value_counts()
df = df[df["User-ID"].isin(active_users[active_users > 30].index)]

popular_books = df["ISBN"].value_counts()
df = df[df["ISBN"].isin(popular_books[popular_books > 50].index)]


In [189]:
popularity_df = (
    df.groupby("ISBN")
    .agg(rating_count=("Book-Rating", "count"),
         avg_rating=("Book-Rating", "mean"))
    .sort_values(["rating_count", "avg_rating"], ascending=False)
    .reset_index()
)


In [190]:
isbn_user_matrix = df.pivot_table(
    index="ISBN",
    columns="User-ID",
    values="Book-Rating"
).fillna(0)

item_similarity = cosine_similarity(isbn_user_matrix)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=isbn_user_matrix.index,
    columns=isbn_user_matrix.index
)


In [191]:
user_isbn_matrix = df.pivot_table(
    index="User-ID",
    columns="ISBN",
    values="Book-Rating"
).fillna(0)

user_similarity = cosine_similarity(user_isbn_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_isbn_matrix.index,
    columns=user_isbn_matrix.index
)


In [192]:
book_details = books[[
    "ISBN",
    "Book-Title",
    "Book-Author",
    "Publisher",
    "Year-Of-Publication",
    "Image-URL-L"
]].drop_duplicates(subset="ISBN")
book_details.to_csv("book_details.csv", index=False)

In [193]:
pickle.dump(item_similarity_df, open("item_model.pkl", "wb"))
pickle.dump(user_similarity_df, open("user_model.pkl", "wb"))
pickle.dump(user_isbn_matrix, open("user_isbn_matrix.pkl", "wb"))

book_details.to_csv("book_details.csv", index=False)
popularity_df.to_csv("popularity.csv", index=False)
users.to_csv("users.csv", index=False)


In [194]:
print("BOOKS COLUMNS:", books.columns.tolist())


BOOKS COLUMNS: ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']


In [195]:
from google.colab import files
files.download("book_details.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>